In [1]:
!pip install selenium webdriver-manager pandas

In [4]:
import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ==================== KONFIGURASI ====================

SEARCH_QUERIES = [
    "coffee shop Semarang Tengah",
    "coffee shop Semarang Barat",
    "coffee shop Semarang Timur",
    "coffee shop Semarang Selatan",
    "coffee shop Semarang Utara",
    "coffee shop Banyumanik Semarang",
    "coffee shop Tembalang Semarang",
    "coffee shop Ngaliyan Semarang",
]

MAX_SCROLL_PER_QUERY = 15
MAX_REVIEWS_PER_CAFE = 500  # target ulasan per cafe. Kalau cafe punya ulasan < 500, otomatis diambil semua (berhenti karena stagnan)
MAX_SCROLL_ATTEMPTS_REVIEW = 400  # jaring pengaman jumlah percobaan scroll
STAGNANT_LIMIT = 6  # berhenti kalau sekian kali scroll berturut-turut tidak menambah ulasan baru
MAX_SECONDS_PER_CAFE = 600  # jaring pengaman waktu: maksimal 10 menit per cafe

CAFE_FILE = "scraping_file_cafe_lengkap.csv"
REVIEWS_FILE = "scraping_file_google_maps_reviews.csv"


# ==================== SETUP ====================

def setup_driver():
    options = Options()
    options.add_argument("--lang=id-ID")
    options.add_argument("--disable-notifications")
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)


def handle_consent_dialog(driver):
    possible_texts = ["Terima semua", "Accept all", "Setuju", "I agree", "Saya setuju"]
    for text in possible_texts:
        try:
            btn = driver.find_element(By.XPATH, f"//button[contains(., '{text}')]")
            btn.click()
            print(f"  [i] Dialog consent ditemukan dan diklik ('{text}')")
            time.sleep(1.5)
            return True
        except Exception:
            continue
    return False


# ==================== TAHAP 1: Kumpulkan daftar cafe ====================

def search_cafes(driver, query):
    print(f"\nMencari area: {query}")
    url = f"https://www.google.com/maps/search/{query.replace(' ', '+')}"
    driver.get(url)
    time.sleep(random.uniform(3, 5))
    handle_consent_dialog(driver)

    try:
        feed = driver.find_element(By.CSS_SELECTOR, "div[role='feed']")
    except Exception:
        print("  [!] Panel hasil pencarian tidak ditemukan, skip.")
        return []

    for _ in range(MAX_SCROLL_PER_QUERY):
        driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", feed)
        time.sleep(random.uniform(1.5, 2.5))

    result_links = driver.find_elements(By.CSS_SELECTOR, "a.hfpxzc")
    cafes = []
    for el in result_links:
        nama = el.get_attribute("aria-label")
        link = el.get_attribute("href")
        if nama and link:
            cafes.append({"nama_cafe": nama, "link": link})

    print(f"  -> ditemukan {len(cafes)} cafe di area ini")
    return cafes


def collect_all_cafe_links(driver):
    all_cafes = {}
    for query in SEARCH_QUERIES:
        for c in search_cafes(driver, query):
            all_cafes[c["link"]] = c["nama_cafe"]
        time.sleep(random.uniform(3, 6))

    print(f"\nTotal cafe unik ditemukan: {len(all_cafes)}")
    return all_cafes


# ==================== TAHAP 2: Rating, price level, alamat, jam operasional ====================

def extract_cafe_summary(driver):
    summary = {
        "rating_rata2": None, "jumlah_ulasan_total": None,
        "harga_indikasi": None, "alamat": None, "jam_operasional": None
    }

    try:
        rating_el = driver.find_element(By.CSS_SELECTOR, "div.F7nice span[aria-hidden='true']")
        summary["rating_rata2"] = rating_el.text.strip()
    except Exception:
        print("  [!] rating_rata2 tidak ditemukan")

    try:
        jumlah_el = driver.find_element(
            By.CSS_SELECTOR,
            "div.F7nice span[aria-label*='ulasan'], div.F7nice span[aria-label*='review']"
        )
        angka = re.sub(r"[^\d]", "", jumlah_el.get_attribute("aria-label") or jumlah_el.text)
        summary["jumlah_ulasan_total"] = angka if angka else None
    except Exception:
        print("  [!] jumlah_ulasan_total tidak ditemukan")

    try:
        price_el = driver.find_elements(By.XPATH, "//span[contains(text(),'Rp') or contains(text(),'$')]")
        if price_el:
            summary["harga_indikasi"] = price_el[0].text.strip()
    except Exception:
        pass

    try:
        alamat_el = driver.find_element(By.XPATH, "//button[contains(@aria-label, 'Alamat')]")
        label = alamat_el.get_attribute("aria-label") or ""
        summary["alamat"] = label.replace("Alamat: ", "").strip()
    except Exception:
        print("  [!] alamat tidak ditemukan")

    jam_ditemukan = False
    hari_list = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"]
    for hari in hari_list:
        try:
            found = driver.find_elements(By.XPATH, f"//*[contains(@aria-label, '{hari}')]")
            if found:
                summary["jam_operasional"] = (found[0].get_attribute("aria-label") or "").strip()
                jam_ditemukan = True
                break
        except Exception:
            continue

    if not jam_ditemukan:
        try:
            found = driver.find_elements(
                By.XPATH,
                "//*[contains(text(),'Buka') or contains(text(),'Tutup') or contains(text(),'24 jam')]"
            )
            if found:
                teks = found[0].text.strip()
                if teks:
                    summary["jam_operasional"] = teks
                    jam_ditemukan = True
        except Exception:
            pass

    if not jam_ditemukan:
        print("  [!] jam_operasional tidak ditemukan")

    return summary


# ==================== TAHAP 3: Ulasan + rating per ulasan ====================

def click_review_tab(driver, timeout=8):
    try:
        tab = WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable((
                By.XPATH,
                "//button[contains(@aria-label, 'Ulasan') or contains(@aria-label, 'Reviews')]"
            ))
        )
        tab.click()
        print("  [i] Tab ulasan diklik lewat STRATEGI 1 (aria-label)")
        return True
    except Exception as e:
        print(f"  [debug] Strategi 1 gagal: {type(e).__name__}")

    try:
        tabs = driver.find_elements(By.CSS_SELECTOR, "button[role='tab']")
        print(f"  [debug] Ditemukan {len(tabs)} tombol role='tab': "
              f"{[t.get_attribute('aria-label') for t in tabs]}")
        if len(tabs) >= 2:
            tabs[1].click()
            print("  [i] Tab ulasan diklik lewat STRATEGI 2 (role='tab' index ke-2)")
            return True
    except Exception as e:
        print(f"  [debug] Strategi 2 gagal: {type(e).__name__}: {e}")

    print("  [!] Tab ulasan GAGAL diklik (kedua strategi gagal)")
    return False


def find_scrollable_review_panel(driver):
    """Cari container ulasan yang BENAR-BENAR scrollable (scrollHeight > clientHeight),
    lalu scroll pakai scrollTop langsung -- pola yang sama persis dengan yang sudah
    terbukti jalan di search_cafes() untuk scroll daftar cafe. Ini jauh lebih reliable
    dibanding simulasi wheel/keyboard yang tergantung posisi elemen di viewport."""
    candidate_selectors = [
        "div.m6QErb[aria-label]",
        "div[role='main'] div.m6QErb",
        "div.DxyBCb",
    ]
    for sel in candidate_selectors:
        try:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            for el in els:
                is_scrollable = driver.execute_script(
                    "return arguments[0].scrollHeight > arguments[0].clientHeight + 10;", el
                )
                if is_scrollable:
                    return el
        except Exception:
            continue
    return None


def scroll_reviews_via_panel(driver, target_count):
    """Scroll panel ulasan pakai scrollTop, berhenti kalau:
    - sudah mencapai target_count ulasan, ATAU
    - stagnan (tidak nambah lagi -> berarti cafe ini ulasannya memang < target), ATAU
    - sudah melebihi batas jumlah percobaan / waktu per cafe (jaring pengaman)."""
    panel = find_scrollable_review_panel(driver)
    if panel is None:
        print("    [!] Panel ulasan scrollable tidak ditemukan! Coba ambil apa yang sudah termuat.")
        return driver.find_elements(By.CSS_SELECTOR, "div.jJc9Ad")

    h = driver.execute_script("return arguments[0].scrollHeight;", panel)
    c = driver.execute_script("return arguments[0].clientHeight;", panel)
    print(f"    [debug] Panel ulasan ditemukan. scrollHeight={h}, clientHeight={c}")

    stagnant_count = 0
    last_count = 0
    start_time = time.time()

    for attempt in range(MAX_SCROLL_ATTEMPTS_REVIEW):
        elapsed = time.time() - start_time
        if elapsed > MAX_SECONDS_PER_CAFE:
            print(f"    -> berhenti, sudah {elapsed:.0f} detik di cafe ini (melebihi batas {MAX_SECONDS_PER_CAFE} detik)")
            break

        driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", panel)
        time.sleep(random.uniform(1.2, 2.0))

        current_reviews = driver.find_elements(By.CSS_SELECTOR, "div.jJc9Ad")
        current_count = len(current_reviews)
        print(f"    scroll #{attempt + 1}: {current_count} ulasan termuat ({elapsed:.0f}s)")

        if target_count is not None and current_count >= target_count:
            print(f"    -> berhenti, target {target_count} ulasan tercapai ({current_count} termuat)")
            break

        if current_count == last_count:
            stagnant_count += 1
            if stagnant_count >= STAGNANT_LIMIT:
                print(f"    -> berhenti, tidak ada ulasan baru setelah {STAGNANT_LIMIT}x scroll "
                      f"(total: {current_count} ulasan -- kemungkinan ini semua ulasan yang tersedia)")
                break
        else:
            stagnant_count = 0

        last_count = current_count

    return driver.find_elements(By.CSS_SELECTOR, "div.jJc9Ad")


def scrape_reviews_from_link(driver, nama_cafe, link):
    print(f"\nMembuka: {nama_cafe}")
    driver.get(link)

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "h1"))
        )
    except Exception:
        print("  [!] Halaman cafe tidak termuat sempurna dalam 10 detik, tetap lanjut coba.")

    handle_consent_dialog(driver)

    summary = extract_cafe_summary(driver)
    print(f"  Rating: {summary['rating_rata2']}, Jumlah ulasan: {summary['jumlah_ulasan_total']}, "
          f"Harga: {summary['harga_indikasi']}, Jam: {summary['jam_operasional']}")

    if not click_review_tab(driver):
        return summary, []

    time.sleep(random.uniform(1.5, 2.5))

    initial_check = driver.find_elements(By.CSS_SELECTOR, "div.jJc9Ad")
    print(f"  [debug] Segera setelah klik tab: {len(initial_check)} elemen div.jJc9Ad langsung ditemukan")

    review_blocks = scroll_reviews_via_panel(driver, MAX_REVIEWS_PER_CAFE)

    # Kalau termuat lebih banyak dari target (bisa terjadi karena batch load Google
    # tidak presisi per-item), potong ke target supaya konsisten per cafe.
    if MAX_REVIEWS_PER_CAFE is not None and len(review_blocks) > MAX_REVIEWS_PER_CAFE:
        review_blocks = review_blocks[:MAX_REVIEWS_PER_CAFE]

    reviews = []
    for block in review_blocks:
        try:
            teks_el = block.find_element(By.CSS_SELECTOR, "span.wiI7pd")
            teks = teks_el.text.strip()
        except Exception:
            teks = None

        try:
            rating_el = block.find_element(By.CSS_SELECTOR, "span[role='img']")
            rating_label = rating_el.get_attribute("aria-label") or ""
            angka_rating = re.search(r"[\d.,]+", rating_label)
            rating_bintang = angka_rating.group().replace(",", ".") if angka_rating else None
        except Exception:
            rating_bintang = None

        if teks:
            reviews.append({"ulasan": teks, "rating_bintang": rating_bintang})

    print(f"  -> berhasil ambil {len(reviews)} ulasan (target: {MAX_REVIEWS_PER_CAFE})")
    return summary, reviews


# ==================== MAIN ====================

def main():
    driver = setup_driver()
    cafe_rows = []
    review_rows = []

    try:
        driver.get("https://www.google.com/maps")
        time.sleep(2)
        handle_consent_dialog(driver)

        print("=== TAHAP 1: Mengumpulkan daftar cafe ===")
        cafe_links = collect_all_cafe_links(driver)

        print("\n=== TAHAP 2 & 3: Rating, price level, alamat, jam, dan ulasan tiap cafe ===")
        for i, (link, nama_cafe) in enumerate(cafe_links.items()):
            summary, reviews = scrape_reviews_from_link(driver, nama_cafe, link)

            cafe_rows.append({
                "nama_cafe": nama_cafe,
                "link": link,
                "alamat": summary["alamat"],
                "rating_rata2": summary["rating_rata2"],
                "jumlah_ulasan_total": summary["jumlah_ulasan_total"],
                "harga_indikasi": summary["harga_indikasi"],
                "jam_operasional": summary["jam_operasional"],
            })

            for r in reviews:
                review_rows.append({
                    "nama_cafe": nama_cafe,
                    "ulasan": r["ulasan"],
                    "rating_bintang": r["rating_bintang"],
                })

            jeda = random.uniform(8, 15)
            print(f"  Jeda {jeda:.1f} detik... ({i + 1}/{len(cafe_links)})")
            time.sleep(jeda)
    finally:
        driver.quit()

    df_cafe = pd.DataFrame(cafe_rows)
    df_cafe.to_csv(CAFE_FILE, index=False, encoding="utf-8-sig")

    df_reviews = pd.DataFrame(review_rows)
    df_reviews.to_csv(REVIEWS_FILE, index=False, encoding="utf-8-sig")

    print(f"\n=== SELESAI ===")
    print(f"Data cafe: {len(df_cafe)} -> {CAFE_FILE}")
    print(f"Data ulasan: {len(df_reviews)} -> {REVIEWS_FILE}")

    return df_cafe, df_reviews


df_cafe, df_reviews = main()

=== TAHAP 1: Mengumpulkan daftar cafe ===

Mencari area: coffee shop Semarang Tengah
  -> ditemukan 80 cafe di area ini

Mencari area: coffee shop Semarang Barat
  -> ditemukan 80 cafe di area ini

Mencari area: coffee shop Semarang Timur
  -> ditemukan 80 cafe di area ini

Mencari area: coffee shop Semarang Selatan
  -> ditemukan 80 cafe di area ini

Mencari area: coffee shop Semarang Utara
  -> ditemukan 80 cafe di area ini

Mencari area: coffee shop Banyumanik Semarang
  -> ditemukan 70 cafe di area ini

Mencari area: coffee shop Tembalang Semarang
  -> ditemukan 80 cafe di area ini

Mencari area: coffee shop Ngaliyan Semarang
  -> ditemukan 70 cafe di area ini

Total cafe unik ditemukan: 446

=== TAHAP 2 & 3: Rating, price level, alamat, jam, dan ulasan tiap cafe ===

Membuka: Eastman Coffee House
  Rating: 4,7, Jumlah ulasan: 4396, Harga: Rp 50–100 rb, Jam: Senin,07.00 hingga 23.00, Salin jam buka
  [i] Tab ulasan diklik lewat STRATEGI 1 (aria-label)
  [debug] Segera setelah klik 

In [1]:
pip install pandas rapidfuzz --break-system-packages

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------------- ---------------------- 0.8/1.8 MB 2.6 MB/s eta 0:00:01
   ----------------------------- ---------- 1.3/1.8 MB 2.2 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 2.3 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
"""
SCRAPING ULANG - khusus cafe dengan cakupan ulasan rendah/gagal total.

Membaca daftar target dari cek_cakupan_ulasan.csv (hasil validasi sebelumnya),
memilih cafe dengan rating_rata2 kosong (gagal total) ATAU persentase_cakupan < 10%,
lalu scraping ulang KHUSUS cafe-cafe itu saja -- bukan full 440 cafe lagi.

PERBAIKAN dari versi sebelumnya:
- keyboard_scroll() sekarang klik ke TOMBOL TAB ULASAN sebagai titik fokus,
  BUKAN ke blok ulasan itu sendiri -- karena blok ulasan berisi tombol link ke
  profil reviewer yang bisa ke-klik tidak sengaja dan mengganggu proses loading.

CARA PAKAI:
1. Pastikan cek_cakupan_ulasan.csv ada di folder yang sama
2. pip install selenium webdriver-manager pandas
3. Jalankan: python retry_scrape.py
4. Hasil: retry_cafe.csv dan retry_reviews.csv
"""

import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.actions.wheel_input import ScrollOrigin
from webdriver_manager.chrome import ChromeDriverManager

CEK_FILE = "cek_cakupan_ulasan.csv"
RETRY_CAFE_FILE = "retry_cafe.csv"
RETRY_REVIEWS_FILE = "retry_reviews.csv"

MAX_REVIEWS_PER_CAFE = 500
MAX_SCROLL_ATTEMPTS_REVIEW = 400
STAGNANT_LIMIT = 6
MAX_SECONDS_PER_CAFE = 900


# ==================== SETUP ====================

def setup_driver():
    options = Options()
    options.add_argument("--lang=id-ID")
    options.add_argument("--disable-notifications")
    # PENTING: sembunyikan fingerprint otomasi -- Google kemungkinan mendeteksi
    # navigator.webdriver=true dan menyajikan UI berbeda (versi dengan tombol
    # "muat lebih banyak" alih-alih infinite scroll) ke browser yang terdeteksi bot.
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument("--window-size=1366,900")  # ukuran window wajar, bukan default aneh

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)

    # Timpa navigator.webdriver jadi undefined SEBELUM halaman apa pun dimuat
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    })
    return driver


def handle_consent_dialog(driver):
    possible_texts = ["Terima semua", "Accept all", "Setuju", "I agree", "Saya setuju"]
    for text in possible_texts:
        try:
            btn = driver.find_element(By.XPATH, f"//button[contains(., '{text}')]")
            btn.click()
            print(f"  [i] Dialog consent ditemukan dan diklik ('{text}')")
            time.sleep(1.5)
            return True
        except Exception:
            continue
    return False


# ==================== RATING, PRICE, ALAMAT, JAM ====================

def extract_cafe_summary(driver):
    summary = {
        "rating_rata2": None, "jumlah_ulasan_total": None,
        "harga_indikasi": None, "alamat": None, "jam_operasional": None
    }
    try:
        rating_el = driver.find_element(By.CSS_SELECTOR, "div.F7nice span[aria-hidden='true']")
        summary["rating_rata2"] = rating_el.text.strip()
    except Exception:
        print("  [!] rating_rata2 tidak ditemukan")

    try:
        jumlah_el = driver.find_element(
            By.CSS_SELECTOR,
            "div.F7nice span[aria-label*='ulasan'], div.F7nice span[aria-label*='review']"
        )
        angka = re.sub(r"[^\d]", "", jumlah_el.get_attribute("aria-label") or jumlah_el.text)
        summary["jumlah_ulasan_total"] = angka if angka else None
    except Exception:
        print("  [!] jumlah_ulasan_total tidak ditemukan")

    try:
        price_el = driver.find_elements(By.XPATH, "//span[contains(text(),'Rp') or contains(text(),'$')]")
        if price_el:
            summary["harga_indikasi"] = price_el[0].text.strip()
    except Exception:
        pass

    try:
        alamat_el = driver.find_element(By.XPATH, "//button[contains(@aria-label, 'Alamat')]")
        label = alamat_el.get_attribute("aria-label") or ""
        summary["alamat"] = label.replace("Alamat: ", "").strip()
    except Exception:
        print("  [!] alamat tidak ditemukan")

    jam_ditemukan = False
    hari_list = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"]
    for hari in hari_list:
        try:
            found = driver.find_elements(By.XPATH, f"//*[contains(@aria-label, '{hari}')]")
            if found:
                summary["jam_operasional"] = (found[0].get_attribute("aria-label") or "").strip()
                jam_ditemukan = True
                break
        except Exception:
            continue
    if not jam_ditemukan:
        try:
            found = driver.find_elements(
                By.XPATH,
                "//*[contains(text(),'Buka') or contains(text(),'Tutup') or contains(text(),'24 jam')]"
            )
            if found and found[0].text.strip():
                summary["jam_operasional"] = found[0].text.strip()
        except Exception:
            pass

    return summary


# ==================== ULASAN (dengan perbaikan keyboard_scroll) ====================

def click_review_tab(driver, timeout=8):
    """Sekarang MENGEMBALIKAN elemen tab-nya juga, supaya bisa dipakai lagi
    sebagai titik fokus keyboard yang AMAN (bukan blok ulasan)."""
    try:
        tab = WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable((
                By.XPATH,
                "//button[contains(@aria-label, 'Ulasan') or contains(@aria-label, 'Reviews')]"
            ))
        )
        tab.click()
        print("  [i] Tab ulasan diklik lewat STRATEGI 1 (aria-label)")
        return True, tab
    except Exception as e:
        print(f"  [debug] Strategi 1 gagal: {type(e).__name__}")

    try:
        tabs = driver.find_elements(By.CSS_SELECTOR, "button[role='tab']")
        if len(tabs) >= 2:
            tabs[1].click()
            print("  [i] Tab ulasan diklik lewat STRATEGI 2 (role='tab' index ke-2)")
            return True, tabs[1]
    except Exception as e:
        print(f"  [debug] Strategi 2 gagal: {type(e).__name__}: {e}")

    print("  [!] Tab ulasan GAGAL diklik (kedua strategi gagal)")
    return False, None


def cdp_wheel_scroll(driver, anchor_element, delta_y=700, ticks=3):
    """Teknik BARU: kirim event wheel lewat Chrome DevTools Protocol (CDP)
    langsung ke koordinat piksel di layar -- ini lebih 'rendah level' dibanding
    ActionChains Selenium biasa, dan tidak bergantung pada elemen mana yang
    'dianggap' scrollable oleh WebDriver. Dipakai kalau wheel & keyboard biasa
    sama-sama gagal."""
    try:
        if anchor_element is not None:
            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center', behavior: 'instant'});",
                anchor_element
            )
            time.sleep(0.3)
            rect = anchor_element.rect
            x = rect["x"] + rect["width"] / 2
            y = rect["y"] + rect["height"] / 2
        else:
            x, y = 400, 400

        for _ in range(ticks):
            driver.execute_cdp_cmd("Input.dispatchMouseEvent", {
                "type": "mouseWheel",
                "x": x, "y": y,
                "deltaX": 0, "deltaY": delta_y,
            })
            time.sleep(random.uniform(0.4, 0.7))
        return True
    except Exception as e:
        print(f"    [debug] cdp_wheel_scroll gagal: {type(e).__name__}: {e}")
        return False


def keyboard_scroll(driver, safe_focus_element, presses=4):
    """DIPERBAIKI: klik ke tombol TAB ULASAN (aman, idempotent), BUKAN ke blok
    ulasan -- supaya tidak ada risiko ke-klik tombol profil reviewer di dalamnya
    yang bisa mengganggu proses loading ulasan berikutnya."""
    try:
        if safe_focus_element is not None:
            ActionChains(driver).move_to_element(safe_focus_element).click().perform()
        for _ in range(presses):
            ActionChains(driver).send_keys(Keys.PAGE_DOWN).perform()
            time.sleep(random.uniform(0.5, 0.8))
        return True
    except Exception as e:
        print(f"    [debug] keyboard_scroll gagal: {type(e).__name__}: {e}")
        return False


def real_wheel_scroll(driver, anchor_element, amount=400, ticks=3):
    try:
        if anchor_element is not None:
            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center', behavior: 'instant'});",
                anchor_element
            )
            time.sleep(0.3)
            origin = ScrollOrigin.from_element(anchor_element)
        else:
            origin = ScrollOrigin.from_viewport(400, 400)
        for _ in range(ticks):
            ActionChains(driver).scroll_from_origin(origin, 0, amount).perform()
            time.sleep(random.uniform(0.4, 0.7))
        return True
    except Exception as e:
        print(f"    [debug] real_wheel_scroll gagal: {type(e).__name__}: {e}")
        try:
            driver.execute_script("window.scrollBy(0, 400);")
        except Exception:
            pass
        return False


def click_load_more_if_present(driver):
    """DIPERBAIKI: kata kunci 'Lainnya' saja ternyata terlalu umum -- kena
    tombol 'baca selengkapnya' di DALAM satu ulasan (memperluas teks
    terpotong), bukan tombol muat ulasan baru. Sekarang pakai frasa lebih
    spesifik DAN mengecualikan tombol yang berada di dalam blok ulasan
    (div.jJc9Ad), supaya tidak salah sasaran lagi."""
    keywords = ["ulasan lainnya", "lebih banyak ulasan", "muat ulasan", "more reviews", "tampilkan ulasan"]
    for kw in keywords:
        try:
            btn = driver.find_element(
                By.XPATH,
                f"//button[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', "
                f"'abcdefghijklmnopqrstuvwxyz'), '{kw}')][not(ancestor::div[contains(@class,'jJc9Ad')])]"
            )
            btn.click()
            print(f"    [i] Tombol '{kw}' ditemukan dan diklik (di luar blok ulasan)")
            return True
        except Exception:
            continue
    return False


def scroll_reviews(driver, tab_element):
    """anchor untuk WHEEL tetap blok ulasan terakhir (posisi harus presisi),
    tapi anchor untuk KEYBOARD sekarang pakai tab_element (aman)."""
    stagnant_count = 0
    last_count = 0
    start_time = time.time()

    for attempt in range(MAX_SCROLL_ATTEMPTS_REVIEW):
        elapsed = time.time() - start_time
        if elapsed > MAX_SECONDS_PER_CAFE:
            print(f"    -> berhenti, sudah {elapsed:.0f} detik (melebihi batas {MAX_SECONDS_PER_CAFE}s)")
            break

        current_reviews = driver.find_elements(By.CSS_SELECTOR, "div.jJc9Ad")
        wheel_anchor = current_reviews[-1] if current_reviews else None

        # Cek dulu apakah ada tombol "muat lebih banyak" -- kalau ada dan berhasil
        # diklik, itu jauh lebih pasti daripada scroll dalam bentuk apa pun.
        tombol_diklik = click_load_more_if_present(driver)

        if tombol_diklik:
            time.sleep(1.0)  # beri waktu render kalau memang tombol asli, lanjut tetap scroll di bawah
        if stagnant_count < 2:
            real_wheel_scroll(driver, wheel_anchor, amount=400, ticks=3)
        elif stagnant_count < 4:
            cdp_wheel_scroll(driver, wheel_anchor, delta_y=700, ticks=3)
        else:
            keyboard_scroll(driver, tab_element, presses=4)  # <-- pakai tab, bukan blok ulasan

        time.sleep(random.uniform(1.0, 1.8))

        current_reviews = driver.find_elements(By.CSS_SELECTOR, "div.jJc9Ad")
        current_count = len(current_reviews)
        print(f"    scroll #{attempt+1}: {current_count} ulasan termuat ({elapsed:.0f}s)")

        if current_count >= MAX_REVIEWS_PER_CAFE:
            print(f"    -> berhenti, target {MAX_REVIEWS_PER_CAFE} tercapai")
            break

        if current_count == last_count:
            stagnant_count += 1
            if stagnant_count >= STAGNANT_LIMIT:
                print(f"    -> berhenti, tidak ada ulasan baru setelah {STAGNANT_LIMIT}x scroll")
                break
        else:
            stagnant_count = 0
        last_count = current_count

    return driver.find_elements(By.CSS_SELECTOR, "div.jJc9Ad")


def scrape_one_cafe(driver, nama_cafe, link):
    print(f"\nMembuka: {nama_cafe}")

    halaman_termuat = False
    for percobaan in range(2):  # coba 2x kalau gagal load di percobaan pertama
        driver.get(link)
        try:
            WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
            halaman_termuat = True
            break
        except Exception:
            print(f"  [!] Percobaan ke-{percobaan+1}: halaman tidak termuat dalam 20 detik.")
            print(f"  [debug] URL saat ini : {driver.current_url}")
            print(f"  [debug] Judul halaman: {driver.title}")
            if percobaan == 0:
                print("  [i] Mencoba reload sekali lagi...")
                time.sleep(3)

    if not halaman_termuat:
        # Upaya terakhir: link lama mungkin sudah "basi" di sisi Google, coba
        # cari ulang lewat NAMA (bukan link tersimpan) sebelum benar-benar menyerah.
        print("  [i] Mencoba cari ulang lewat nama (link lama mungkin sudah tidak valid)...")
        try:
            search_url = f"https://www.google.com/maps/search/{nama_cafe.replace(' ', '+')}+Semarang"
            driver.get(search_url)
            time.sleep(4)
            first_result = driver.find_elements(By.CSS_SELECTOR, "a.hfpxzc")
            if first_result:
                first_result[0].click()
                WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
                halaman_termuat = True
                print("  [i] Berhasil ditemukan lewat pencarian ulang nama.")
        except Exception as e:
            print(f"  [debug] Pencarian ulang lewat nama juga gagal: {type(e).__name__}")

    if not halaman_termuat:
        try:
            nama_file_aman = "".join(c if c.isalnum() else "_" for c in nama_cafe)[:50]
            screenshot_path = f"debug_gagal_{nama_file_aman}.png"
            driver.save_screenshot(screenshot_path)
            print(f"  [debug] Screenshot disimpan: {screenshot_path} (cek manual untuk lihat kenapa gagal)")
        except Exception:
            pass
        return {"rating_rata2": None, "jumlah_ulasan_total": None, "harga_indikasi": None,
                "alamat": None, "jam_operasional": None}, []

    handle_consent_dialog(driver)

    summary = extract_cafe_summary(driver)
    print(f"  Rating: {summary['rating_rata2']}, Jumlah ulasan: {summary['jumlah_ulasan_total']}, "
          f"Harga: {summary['harga_indikasi']}, Jam: {summary['jam_operasional']}")

    berhasil, tab_element = click_review_tab(driver)
    if not berhasil:
        return summary, []

    time.sleep(random.uniform(1.5, 2.5))
    review_blocks = scroll_reviews(driver, tab_element)

    reviews = []
    for block in review_blocks[:MAX_REVIEWS_PER_CAFE]:
        try:
            teks = block.find_element(By.CSS_SELECTOR, "span.wiI7pd").text.strip()
        except Exception:
            teks = None
        try:
            rating_el = block.find_element(By.CSS_SELECTOR, "span[role='img']")
            label = rating_el.get_attribute("aria-label") or ""
            m = re.search(r"[\d.,]+", label)
            rating_bintang = m.group().replace(",", ".") if m else None
        except Exception:
            rating_bintang = None
        if teks:
            reviews.append({"ulasan": teks, "rating_bintang": rating_bintang})

    print(f"  -> berhasil ambil {len(reviews)} ulasan")
    return summary, reviews


# ==================== MAIN ====================

def main():
    df_cek = pd.read_csv(CEK_FILE)
    target = df_cek[df_cek["rating_rata2"].isna() | (df_cek["persentase_cakupan"] < 10)]
    print(f"Total cafe yang akan di-scraping ulang: {len(target)}")
    print(target[["nama_cafe", "jumlah_ulasan_total", "ulasan_terambil"]].to_string())

    driver = setup_driver()
    cafe_rows = []
    review_rows = []

    try:
        driver.get("https://www.google.com/maps")
        time.sleep(2)
        handle_consent_dialog(driver)

        for i, row in enumerate(target.itertuples(), start=1):
            print(f"\n=== [{i}/{len(target)}] {row.nama_cafe} ===")
            summary, reviews = scrape_one_cafe(driver, row.nama_cafe, row.link)

            cafe_rows.append({
                "nama_cafe": row.nama_cafe,
                "link": row.link,
                "alamat": summary["alamat"],
                "rating_rata2": summary["rating_rata2"],
                "jumlah_ulasan_total": summary["jumlah_ulasan_total"],
                "harga_indikasi": summary["harga_indikasi"],
                "jam_operasional": summary["jam_operasional"],
            })
            for r in reviews:
                review_rows.append({
                    "nama_cafe": row.nama_cafe,
                    "ulasan": r["ulasan"],
                    "rating_bintang": r["rating_bintang"],
                })

            jeda = random.uniform(8, 15)
            print(f"  Jeda {jeda:.1f} detik...")
            time.sleep(jeda)
    finally:
        driver.quit()

    pd.DataFrame(cafe_rows).to_csv(RETRY_CAFE_FILE, index=False, encoding="utf-8-sig")
    pd.DataFrame(review_rows).to_csv(RETRY_REVIEWS_FILE, index=False, encoding="utf-8-sig")

    print(f"\n=== SELESAI ===")
    print(f"Cafe hasil scraping ulang   -> {RETRY_CAFE_FILE}")
    print(f"Ulasan hasil scraping ulang -> {RETRY_REVIEWS_FILE}")
    print("\nLangkah selanjutnya: jalankan merge_retry.py untuk menggabungkan hasil ini ke data utama.")


if __name__ == "__main__":
    main()

Total cafe yang akan di-scraping ulang: 19
                               nama_cafe  jumlah_ulasan_total  ulasan_terambil
56                     YiSan Coffee Shop                  NaN                0
125                       EL LOCO COFFEE                 15.0                0
161   Ideologist Coffee And Social Space               5254.0              500
168                            coffeshop                  1.0                0
170           Keizai Coffee & Garden MD3                530.0               30
174                         CERITA.COFFE                  NaN                0
188              Dapur Mama Cafe & Resto               8776.0              500
197               Pelangi cafe singosari              10523.0              500
233                        Lajoer Coffee                  2.0                0
344                          Tawa Coffee                  2.0                0
352                    Mbah Imam / Smith                  1.0                0
368      

In [3]:
"""
MERGE RETRY - Menggabungkan hasil scraping ulang (retry_cafe.csv, retry_reviews.csv)
ke data mentah utama (file_cafe_lengkap.csv, file_google_maps_reviews.csv).

Setelah ini selesai, jalankan ULANG step3_cleaning.py dan validasi_data.py dari awal
-- supaya data yang sudah digabung diproses lagi secara konsisten (dedup, cleaning),
bukan cuma ditambal manual.

CARA PAKAI:
Taruh file ini satu folder dengan file_cafe_lengkap.csv, file_google_maps_reviews.csv,
retry_cafe.csv, retry_reviews.csv. Jalankan: python merge_retry.py
"""

import pandas as pd

df_cafe_lama = pd.read_csv("scraping_file_cafe_lengkap.csv")
df_reviews_lama = pd.read_csv("scraping_file_google_maps_reviews.csv")
df_cafe_retry = pd.read_csv("retry_cafe.csv")
df_reviews_retry = pd.read_csv("retry_reviews.csv")

print(f"Data cafe lama       : {len(df_cafe_lama)}")
print(f"Data cafe hasil retry: {len(df_cafe_retry)}")

# --- Gabungkan data cafe: baris retry MENGGANTIKAN baris lama untuk nama_cafe yang sama ---
nama_di_retry = set(df_cafe_retry["nama_cafe"])
df_cafe_gabungan = df_cafe_lama[~df_cafe_lama["nama_cafe"].isin(nama_di_retry)]
df_cafe_gabungan = pd.concat([df_cafe_gabungan, df_cafe_retry], ignore_index=True)
print(f"Data cafe setelah digabung: {len(df_cafe_gabungan)} (harus sama dengan jumlah cafe lama)")

# --- Gabungkan data ulasan: buang ulasan lama milik cafe yang di-retry, ganti dengan yang baru ---
df_reviews_gabungan = df_reviews_lama[~df_reviews_lama["nama_cafe"].isin(nama_di_retry)]
df_reviews_gabungan = pd.concat([df_reviews_gabungan, df_reviews_retry], ignore_index=True)
print(f"Data ulasan setelah digabung: {len(df_reviews_gabungan)} (dari {len(df_reviews_lama)} lama)")

# --- Simpan, menimpa file mentah utama (backup dulu file lama secara manual kalau mau aman) ---
df_cafe_gabungan.to_csv("file_cafe_lengkap.csv", index=False, encoding="utf-8-sig")
df_reviews_gabungan.to_csv("file_google_maps_reviews.csv", index=False, encoding="utf-8-sig")

print("\n=== SELESAI ===")
print("file_cafe_lengkap.csv dan file_google_maps_reviews.csv sudah diperbarui.")
print("Langkah selanjutnya: jalankan ULANG step3_cleaning.py, lalu validasi_data.py.")

Data cafe lama       : 446
Data cafe hasil retry: 19
Data cafe setelah digabung: 446 (harus sama dengan jumlah cafe lama)
Data ulasan setelah digabung: 72915 (dari 72906 lama)

=== SELESAI ===
file_cafe_lengkap.csv dan file_google_maps_reviews.csv sudah diperbarui.
Langkah selanjutnya: jalankan ULANG step3_cleaning.py, lalu validasi_data.py.


In [1]:
"""
SCRIPT GABUNGAN - Menjalankan 3 tahap sekaligus secara berurutan:
1. Merge hasil retry_scrape ke data mentah utama (file_cafe_lengkap.csv, file_google_maps_reviews.csv)
2. Cleaning & deduplikasi + gabung atribut dari OSM (osm_raw.csv) -> cafe_clean.csv, reviews_clean.csv
3. Validasi kualitas data (cetak laporan + simpan cek_cakupan_ulasan.csv)

Perbaikan versi ini:
- Emoji diganti SPASI (bukan dihapus), supaya "enak😍yg" tidak menyambung jadi "enakyg".
- Kamus normalisasi diperluas + menangani huruf berulang ("ygg", "gaaak") dan
  kata ulang berangka ("orang2" -> "orang orang").
- Ada self-test normalisasi, laporan singkatan yang masih tersisa, dan laporan
  token pendek paling sering (untuk mencari singkatan baru secara iteratif).
- Enter/newline di dalam teks ulasan dirapikan jadi spasi (mencegah baris CSV "rusak").

CARA PAKAI:
Taruh semua file berikut di folder yang sama, lalu jalankan:
  - file_cafe_lengkap.csv
  - file_google_maps_reviews.csv
  - osm_raw.csv
  - retry_cafe.csv       (opsional -- kalau tidak ada, tahap ini dilewati)
  - retry_reviews.csv    (opsional -- kalau tidak ada, tahap ini dilewati)

python gabung_bersih_validasi.py
"""

import os
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process
from rapidfuzz.distance import OSA

CAFE_FILE = "file_cafe_lengkap.csv"
REVIEWS_FILE = "file_google_maps_reviews.csv"
OSM_FILE = "osm_raw.csv"
RETRY_CAFE_FILE = "retry_cafe.csv"
RETRY_REVIEWS_FILE = "retry_reviews.csv"

CAFE_CLEAN_OUT = "cafe_clean.csv"
REVIEWS_CLEAN_OUT = "reviews_clean.csv"
CEK_CAKUPAN_OUT = "cek_cakupan_ulasan.csv"
KANDIDAT_TYPO_OUT = "kandidat_typo.csv"

DEDUP_NAME_THRESHOLD = 90
OSM_MATCH_THRESHOLD = 85


# ==================== FUNGSI-FUNGSI BANTU ====================

def normalize_name(name):
    if pd.isna(name):
        return ""
    name = name.lower().strip()
    name = re.sub(r"[^\w\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def extract_lat_lng(link):
    if pd.isna(link):
        return None, None
    match = re.search(r"!3d(-?\d+\.\d+)!4d(-?\d+\.\d+)", link)
    if match:
        return float(match.group(1)), float(match.group(2))
    return None, None


def haversine_km(lat1, lon1, lat2, lon2):
    from math import radians, sin, cos, sqrt, atan2
    R = 6371
    dlat, dlon = radians(lat2 - lat1), radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))


# ---------- Emoji ----------
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticon wajah
    "\U0001F300-\U0001F5FF"  # simbol & piktograf
    "\U0001F680-\U0001F6FF"  # transportasi & peta
    "\U0001F1E0-\U0001F1FF"  # bendera
    "\U00002700-\U000027BF"  # dingbat
    "\U0001F900-\U0001F9FF"  # simbol tambahan
    "\U00002600-\U000026FF"  # simbol lain-lain
    "\U0001FA70-\U0001FAFF"  # simbol tambahan lanjutan
    "\U0001F004\U0001F0CF\U0001F18E"
    "\U0000FE0F"              # variation selector
    "\U0000200D"              # zero-width joiner
    "]+", flags=re.UNICODE
)


# Selain daftar rentang di atas, hapus SEMUA karakter berkategori Unicode "So"
# (Symbol, other: bintang, hati, panah, tanda centang, dll), blok emoji 1F000-1FAFF,
# blok 2B00-2BFF (mis. bintang U+2B50), dan penyambung emoji. Cara ini menangkap
# emoji/simbol yang tidak terdaftar di rentang manual.
_SIMBOL_DIPERTAHANKAN = "\u00b0"   # derajat (25°C) tetap dipertahankan


def _bangun_pola_simbol():
    rentang, mulai, akhir = [], None, None
    for cp in range(0x80, 0x110000):
        if 0xD800 <= cp <= 0xDFFF:
            cocok = False
        else:
            ch = chr(cp)
            cocok = (
                (unicodedata.category(ch) == "So" and ch not in _SIMBOL_DIPERTAHANKAN)
                or 0x1F000 <= cp <= 0x1FAFF
                or 0x2B00 <= cp <= 0x2BFF
                or 0xE0020 <= cp <= 0xE007F
                or cp in (0x200D, 0xFE0F, 0x20E3)
            )
        if cocok:
            if mulai is None:
                mulai = cp
            akhir = cp
        elif mulai is not None:
            rentang.append((mulai, akhir))
            mulai = None
    if mulai is not None:
        rentang.append((mulai, akhir))
    isi = "".join(
        re.escape(chr(a)) if a == b else re.escape(chr(a)) + "-" + re.escape(chr(b))
        for a, b in rentang
    )
    return re.compile("[" + isi + "]+")


POLA_SIMBOL = _bangun_pola_simbol()


def bersihkan_emoji(teks):
    # Emoji diganti SPASI (bukan ""), supaya kata di kiri-kanannya tidak menyambung.
    # \s+ juga merapikan newline/tab di dalam teks ulasan.
    teks = EMOJI_PATTERN.sub(" ", str(teks))
    teks = POLA_SIMBOL.sub(" ", teks)
    teks = re.sub(r"\s+", " ", teks).strip()
    return teks


# ---------- Normalisasi singkatan / kata tidak baku ----------
KAMUS_NORMALISASI = {
    # kata penghubung / umum
    "yg": "yang", "yng": "yang", "dgn": "dengan", "dg": "dengan", "dgan": "dengan",
    "utk": "untuk", "untk": "untuk", "dr": "dari", "krn": "karena", "karna": "karena",
    "krna": "karena", "sm": "sama", "tp": "tapi", "tpi": "tapi",
    "dll": "dan lain lain", "dsb": "dan sebagainya", "dst": "dan seterusnya",
    "pd": "pada", "spt": "seperti", "kyk": "seperti", "sbg": "sebagai",
    "ttg": "tentang", "tsb": "tersebut", "lbh": "lebih", "lbih": "lebih",
    "sll": "selalu", "slalu": "selalu", "spy": "supaya", "sblm": "sebelum",
    "stlh": "setelah", "stelah": "setelah", "smpe": "sampai", "sampe": "sampai",
    "msh": "masih", "masi": "masih", "hny": "hanya", "cm": "cuma", "cuman": "cuma",
    "plg": "paling", "dlm": "dalam", "jgn": "jangan", "bbrp": "beberapa",
    "n": "dan", "ni": "ini", "tu": "itu",
    # negasi
    "tdk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak", "ngga": "tidak",
    "gk": "tidak", "kaga": "tidak", "engga": "tidak", "enggak": "tidak",
    "ngak": "tidak", "gpp": "tidak apa apa", "gapapa": "tidak apa apa",
    "gada": "tidak ada", "blm": "belum", "blum": "belum", "bkn": "bukan",
    # kata ganti
    "sy": "saya", "gw": "saya", "gue": "saya", "gua": "saya",
    "kmu": "kamu", "lu": "kamu", "loe": "kamu",
    # waktu & keadaan
    "udh": "sudah", "udah": "sudah", "uda": "sudah", "sdh": "sudah", "dah": "sudah",
    "skrg": "sekarang", "sekrg": "sekarang", "kmrn": "kemarin", "kemaren": "kemarin",
    "bru": "baru", "lg": "lagi", "lgi": "lagi", "wkt": "waktu", "hbs": "habis",
    "jd": "jadi", "jdi": "jadi", "jg": "juga", "jga": "juga",
    "klo": "kalau", "kalo": "kalau", "kl": "kalau", "gt": "begitu", "gitu": "begitu",
    "gmn": "bagaimana", "gimana": "bagaimana", "knp": "kenapa",
    "aja": "saja", "aj": "saja", "ajah": "saja", "doang": "saja", "sj": "saja",
    # kata kerja / sifat
    "bs": "bisa", "bsa": "bisa", "hrs": "harus", "dpt": "dapat", "dpet": "dapat",
    "dtg": "datang", "liat": "lihat", "lht": "lihat", "pgn": "pengen", "pngen": "pengen",
    "pke": "pakai", "pake": "pakai", "pkai": "pakai", "tau": "tahu", "tw": "tahu",
    "trs": "terus", "trus": "terus", "mnrt": "menurut",
    "emg": "memang", "emng": "memang", "emang": "memang",
    "pdhl": "padahal", "padhal": "padahal", "sptnya": "sepertinya",
    "kayanya": "sepertinya", "kayaknya": "sepertinya", "krg": "kurang", "kurng": "kurang",
    "sndiri": "sendiri", "sndri": "sendiri", "bnr": "benar", "bner": "benar",
    "bgt": "banget", "bngt": "banget", "sgt": "sangat",
    "bgs": "bagus", "bgus": "bagus", "enk": "enak", "mantab": "mantap",
    "asik": "asyik", "cafe": "kafe",   # penyeragaman ejaan (hapus "cafe" kalau tidak mau diseragamkan)
    "mhl": "mahal", "mrh": "murah", "rame": "ramai", "ramee": "ramai",
    "lmyn": "lumayan", "sdkt": "sedikit", "bnyk": "banyak", "byk": "banyak",
    # nomina
    "org": "orang", "tmpt": "tempat", "tmpat": "tempat", "tmn": "teman", "temen": "teman",
    "hrg": "harga", "hrga": "harga", "mkn": "makan", "mknan": "makanan", "mkanan": "makanan",
    "mnm": "minum", "jln": "jalan", "jl": "jalan", "pel": "pelayanan",
    "dket": "dekat", "dkt": "dekat", "dpn": "depan", "blkg": "belakang",
    "kesini": "ke sini", "disini": "di sini", "kesana": "ke sana", "disana": "di sana",
    "pesen": "pesan", "dateng": "datang", "bener": "benar", "nunggu": "menunggu",
    "dapet": "dapat", "sekedar": "sekadar", "kopsu": "kopi susu",
    "aplg": "apalagi",
    # typo umum (tambahkan hasil tinjauan kandidat_typo.csv di sini)
    "wahib": "wajib", "snagat": "sangat",
    "makasih": "terima kasih", "mksh": "terima kasih", "trims": "terima kasih",
    "tks": "terima kasih", "thx": "terima kasih",
    "recomended": "recommended", "rekomended": "recommended", "recom": "recommended",
    "rekomen": "recommended", "reko": "recommended",
}


# Typo yang SUDAH ditinjau manual dari kandidat_typo.csv. Aturannya: hanya kata yang bukan kata sah
# (Indonesia/Inggris) dan punya satu perbaikan yang tidak ambigu. Kata sah seperti "walau", "tepat",
# "kalah", "taman" JANGAN dimasukkan, walau mirip kata lain, karena artinya akan berubah.
KAMUS_TYPO = {
    "tempatnyaa": "tempatnya", "nyamann": "nyaman", "pelayananya": "pelayanannya",
    "suasanya": "suasananya", "mayan": "lumayan", "sekalii": "sekali", "nugass": "nugas",
    "minumanya": "minumannya", "bangeet": "banget", "indor": "indoor", "tempatny":
    "tempatnya", "kopinyaa": "kopinya", "nongky": "nongki", "bnget": "banget", "bnyak":
    "banyak", "kureng": "kurang", "outdor": "outdoor", "mantaps": "mantap", "buanget":
    "banget", "cucok": "cocok", "nongkii": "nongki", "nongrong": "nongkrong", "nyamaan":
    "nyaman", "tempanya": "tempatnya", "pingin": "pengen", "mantaap": "mantap", "kerenn":
    "keren", "makanannyaa": "makanannya", "cofee": "coffee", "pelayanannyaa":
    "pelayanannya", "chiken": "chicken", "trnyata": "ternyata", "teruss": "terus", "skali":
    "sekali", "kereen": "keren", "trlalu": "terlalu", "ramaah": "ramah", "nongkrongg":
    "nongkrong", "temptnya": "tempatnya", "kluarga": "keluarga", "smoga": "semoga",
    "pkoknya": "pokoknya", "makanann": "makanan", "buanyak": "banyak", "ngbrol": "ngobrol",
    "harganyaa": "harganya", "sperti": "seperti", "makananny": "makanannya", "terjangkauu":
    "terjangkau", "ngopii": "ngopi", "tempate": "tempatnya", "banyaak": "banyak", "ngupi":
    "ngopi", "rasanyaa": "rasanya", "semuanyaa": "semuanya", "baget": "banget",
    "makannanya": "makanannya", "kopiny": "kopinya", "superr": "super", "bisaa": "bisa",
    "suasanannya": "suasananya", "instagramble": "instagramable", "kerjaa": "kerja",
    "sdikit": "sedikit", "pelayanann": "pelayanan", "hiden": "hidden", "mantaf": "mantap",
    "apalgi": "apalagi", "keluargaa": "keluarga", "banya": "banyak", "smpai": "sampai",
    "makannannya": "makanannya", "creamyy": "creamy", "outdornya": "outdoornya", "lumayann":
    "lumayan", "menunyaa": "menunya", "tempatmya": "tempatnya", "nonkrong": "nongkrong",
    "suasannya": "suasananya", "asyiik": "asyik", "trmpat": "tempat", "pdahal": "padahal",
    "kmren": "kemarin", "macha": "matcha", "bangt": "banget", "muantap": "mantap",
    "susananya": "suasananya", "cobainn": "cobain", "baristanyaa": "baristanya", "hargaa":
    "harga", "stengah": "setengah", "estetic": "estetik", "semarangg": "semarang",
}
KAMUS_NORMALISASI.update(KAMUS_TYPO)


# Kata yang SUDAH ditinjau dan diputuskan BUKAN typo (kata sah, kata asing, dialek, atau ambigu).
# Disimpan di sini supaya tidak muncul lagi di laporan kandidat typo.
KATA_DIABAIKAN = {
    "tepatnya", "jualan", "anget", "tenan", "orange", "serving", "variant", "rokoknya",
    "susana", "annya", "tempt", "snacks", "bernama", "datangi", "seberapa", "rasakan",
    "menurun", "bersantap", "temani", "tambahi", "merekah", "nerima", "services", "places",
    "pesto", "wcnya", "worthy", "superb", "cheesy", "plate", "psnya", "brasa", "krasa",
    "juarak", "pelayanya",
}


def _cari_token(low):
    """Cari 1 token (huruf kecil) di kamus, termasuk versi huruf berulang ("ygg", "gaaak")."""
    if low in KAMUS_NORMALISASI:
        return KAMUS_NORMALISASI[low]
    if len(low) >= 3:
        squeezed = re.sub(r"(.)\1+", r"\1", low)
        if squeezed in KAMUS_NORMALISASI:
            return KAMUS_NORMALISASI[squeezed]
    return None


def _cari_di_kamus(token):
    low = token.lower()
    hasil = _cari_token(low)
    if hasil is not None:
        return hasil
    # Token berakhiran "nya": "tmptnya" -> "tempatnya", "kayaknya" -> "sepertinya"
    if len(low) > 4 and low.endswith("nya"):
        hasil = _cari_token(low[:-3])
        if hasil is not None:
            return hasil + "nya"
    return None


# "nya" yang terpisah dari kata induknya oleh spasi / tanda baca:
# "enak nya", "Makanan.nya", "service'nya", "Mas\"nya", "\u201ckopsu\u201d nya"
_POLA_NYA = re.compile(r'''(?<=\w)[\s'"\u2019\u2018\u201c\u201d\-\u2013.+]+nya\b''', flags=re.IGNORECASE)


def normalisasi_singkatan(teks):
    teks = str(teks)
    # 1) kata ulang berangka: "orang2" -> "orang orang", "yg2" -> "yg yg" (dibakukan di langkah 4)
    teks = re.sub(r"\b([A-Za-z]+)2\b", r"\1 \1", teks)
    # 2) "t4" = "tempat"
    teks = re.sub(r"\bt4\b", "tempat", teks, flags=re.IGNORECASE)
    # "ny" = "nya" ("kopi v60 ny juga enak"); "d" = "di" ("d smg" = "di semarang")
    teks = re.sub(r"\bny\b", "nya", teks)
    teks = re.sub(r"(?<![\w'\u2019])d(?=\s+[a-z])", "di", teks)
    # "smg" = Semarang HANYA setelah kata depan/tempat ("di smg", "ke smg", "kota smg").
    # "smg" sendirian ambigu (bisa "semoga") sehingga tidak diubah.
    teks = re.sub(r"\b(di|ke|dari|kota|khas)\s+smg\b", r"\1 semarang", teks, flags=re.IGNORECASE)
    # 3) rapatkan "nya" SEBELUM kamus dipakai: "kayak nya" -> "kayaknya" -> "sepertinya"
    teks = _POLA_NYA.sub("nya", teks)

    # 4) ganti per token huruf; token yang tidak ada di kamus dibiarkan
    def ganti(m):
        hasil = _cari_di_kamus(m.group(0))
        return hasil if hasil is not None else m.group(0)

    teks = re.sub(r"[A-Za-z]+", ganti, teks)
    return re.sub(r"\s+", " ", teks).strip()


# ---------- Huruf memanjang ("enakkk", "bangettt", "yaaa") ----------
_RUN3 = re.compile(r"([a-z])\1{2,}")


def bangun_vocab(seri_teks):
    """Hitung frekuensi kata di seluruh ulasan (dipakai untuk memilih ejaan yang benar)."""
    vocab = Counter()
    for u in seri_teks:
        vocab.update(re.findall(r"[a-z]+", str(u).lower()))
    return vocab


def perbaiki_pemanjangan(teks, vocab):
    """
    Huruf berulang >= 3 kali dikecilkan. Pilihan 1 huruf ("goood" -> "god") atau
    2 huruf ("good") ditentukan dari mana yang lebih sering muncul di korpus.
    Huruf ganda di akhir kata (k/t/h/p) seperti "enakk" -> "enak" hanya diubah
    kalau bentuk tanpa gandanya jauh lebih sering (>= 10x), supaya kata asing
    yang memang berhuruf ganda tidak rusak.
    """
    def ganti(m):
        tok = m.group(0)
        low = tok.lower()
        baru = low
        if _RUN3.search(low):
            c1 = _RUN3.sub(r"\1", low)
            c2 = _RUN3.sub(r"\1\1", low)
            baru = c2 if vocab.get(c2, 0) > vocab.get(c1, 0) else c1
        if len(baru) >= 4 and baru[-1] == baru[-2] and baru[-1] in "kthp":
            stem = baru[:-1]
            if vocab.get(stem, 0) >= max(1, 10 * vocab.get(baru, 0)):
                baru = stem
        return tok if baru == low else baru

    return re.sub(r"[A-Za-z]+", ganti, str(teks))


def muat_kamus_kata_dasar():
    """Kata dasar bahasa Indonesia dari paket Sastrawi (opsional: pip install PySastrawi).
    Hanya dipakai untuk MENYARING laporan kandidat typo, tidak pernah untuk mengubah teks."""
    try:
        import Sastrawi
        path = os.path.join(os.path.dirname(Sastrawi.__file__), "Stemmer", "data", "kata-dasar.txt")
        with open(path, encoding="utf-8") as f:
            return {w.strip().lower() for w in f if w.strip()}
    except Exception:
        return set()


def cari_kandidat_typo(vocab, min_typo=2, maks_typo=200, min_benar=100, rasio=20, panjang_min=5,
                       kata_sah=frozenset()):
    """
    Cari kata jarang yang hanya beda 1 huruf (ganti/tambah/hapus/tukar posisi) dari kata
    yang JAUH lebih sering. Hasilnya hanya kandidat untuk ditinjau manual, tidak diubah
    otomatis, karena kata yang benar pun bisa mirip kata lain (mis. "makan" vs "makna").
    """
    benar = [w for w, c in vocab.items() if c >= min_benar and len(w) >= panjang_min]
    jarang = [(w, c) for w, c in vocab.items()
              if min_typo <= c <= maks_typo and len(w) >= panjang_min
              and w not in KAMUS_NORMALISASI and w not in kata_sah and w not in KATA_DIABAIKAN]
    hasil = []
    if not benar or not jarang:
        return hasil
    kata = [w for w, _ in jarang]
    for i in range(0, len(kata), 2000):
        blok = kata[i:i + 2000]
        # matriks jarak (dihitung paralel); nilai 1 = beda tepat 1 huruf
        mat = process.cdist(blok, benar, scorer=OSA.distance, score_cutoff=1,
                            workers=-1, dtype=np.uint8)
        for r, w in enumerate(blok):
            idx = np.nonzero(mat[r] == 1)[0]
            if len(idx) == 0:
                continue
            b, cb = max(((benar[j], vocab[benar[j]]) for j in idx), key=lambda x: x[1])
            c = vocab[w]
            if cb >= rasio * c:
                hasil.append((w, c, b, cb))
    hasil.sort(key=lambda x: -x[1])
    return hasil


def self_test_normalisasi():
    """Tes cepat supaya ketahuan langsung kalau fungsi normalisasi tidak bekerja."""
    vocab_uji = Counter({"banget": 100, "bangett": 5, "enak": 100, "enakk": 3,
                         "good": 50, "god": 1, "ya": 100, "yaa": 5})

    def alur(t):
        return normalisasi_singkatan(perbaiki_pemanjangan(bersihkan_emoji(t), vocab_uji))

    kasus = {
        "Cafe yg eatery nya sm nasi": "kafe yang eaterynya sama nasi",
        "tempatnya asik": "tempatnya asyik",
        "ygg enak, tp muter2 klo bgt": "yang enak, tapi muter muter kalau banget",
        "Pelayanan dr dpn good": "Pelayanan dari depan good",
        "kopi n sarapan": "kopi dan sarapan",
        "gaaak nyangka, emng udah beda": "tidak nyangka, memang sudah beda",
        "harga juga bagus": "harga juga bagus",   # kata baku tidak boleh berubah
        "kayak nya enak, kaya nya oke": "sepertinya enak, sepertinya oke",
        "tmpt nya bgs, t4 nyaman": "tempatnya bagus, tempat nyaman",
        "bangettt enaaak goood yaaa enakk": "banget enak good ya enak",
        "Makanan.nya enak, service'nya jelek": "Makanannya enak, servicenya jelek",
        'Mas"nya good, \u201ckopsu\u201d nya cremy': "Masnya good, \u201ckopi susunya cremy",
        "espresso.nya juga oke": "espressonya juga oke",
        "bintang 5 \u2b50 \u2b50 \u2605 \u2764\ufe0f \U0001f31f \u2728 mantap": "bintang 5 mantap",
        "suhu 25\u00b0C": "suhu 25\u00b0C",   # derajat tidak boleh hilang
        "kopi v60 ny juga enak": "kopi v60nya juga enak",
        "aplg buka d smg": "apalagi buka di semarang",
        "semoga smg tetap enak": "semoga smg tetap enak",   # smg sendirian tidak diubah
        "wahib coba, snagat enak": "wajib coba, sangat enak",
        "nyamann sekalii kerenn, mayan buat nongkrong": "nyaman sekali keren, lumayan buat nongkrong",
        "walau tepat kalah, taman kota": "walau tepat kalah, taman kota",   # kata sah tidak boleh berubah
        "coffee sama latte": "coffee sama latte",  # huruf ganda kata asing tidak boleh rusak
    }
    gagal = 0
    for asal, harapan in kasus.items():
        hasil = alur(asal)
        if hasil != harapan:
            gagal += 1
            print(f"  [GAGAL] '{asal}' -> '{hasil}' (harusnya '{harapan}')")
    print(f"Self-test normalisasi: {len(kasus) - gagal}/{len(kasus)} lulus")
    e = alur("enak😍yg bagus")
    print(f"Self-test emoji: 'enak😍yg bagus' -> '{e}' (harusnya 'enak yang bagus')")


# ==================== TAHAP 1: MERGE HASIL RETRY ====================

print("#" * 70)
print("TAHAP 1: MERGE HASIL RETRY KE DATA MENTAH UTAMA")
print("#" * 70)

df_cafe_raw = pd.read_csv(CAFE_FILE)
df_reviews_raw = pd.read_csv(REVIEWS_FILE)
print(f"Data cafe mentah (sebelum merge retry)   : {len(df_cafe_raw)}")
print(f"Data ulasan mentah (sebelum merge retry) : {len(df_reviews_raw)}")

if os.path.exists(RETRY_CAFE_FILE) and os.path.exists(RETRY_REVIEWS_FILE):
    df_cafe_retry = pd.read_csv(RETRY_CAFE_FILE)
    df_reviews_retry = pd.read_csv(RETRY_REVIEWS_FILE)
    print(f"Ditemukan hasil retry: {len(df_cafe_retry)} cafe, {len(df_reviews_retry)} ulasan")

    nama_di_retry = set(df_cafe_retry["nama_cafe"])

    # Baris lama untuk cafe yang di-retry DIGANTI, bukan ditambah dobel
    df_cafe_raw = df_cafe_raw[~df_cafe_raw["nama_cafe"].isin(nama_di_retry)]
    df_cafe_raw = pd.concat([df_cafe_raw, df_cafe_retry], ignore_index=True)

    df_reviews_raw = df_reviews_raw[~df_reviews_raw["nama_cafe"].isin(nama_di_retry)]
    df_reviews_raw = pd.concat([df_reviews_raw, df_reviews_retry], ignore_index=True)

    # Timpa file mentah utama supaya konsisten untuk run berikutnya
    df_cafe_raw.to_csv(CAFE_FILE, index=False, encoding="utf-8-sig")
    df_reviews_raw.to_csv(REVIEWS_FILE, index=False, encoding="utf-8-sig")

    print(f"Setelah merge retry -> cafe: {len(df_cafe_raw)}, ulasan: {len(df_reviews_raw)}")
    print(f"({CAFE_FILE} dan {REVIEWS_FILE} sudah diperbarui)")
else:
    print("Tidak ditemukan retry_cafe.csv / retry_reviews.csv -- tahap ini dilewati.")

df_cafe = df_cafe_raw.copy()
df_reviews = df_reviews_raw.copy()


# ==================== TAHAP 2: CLEANING & DEDUPLIKASI ====================

print("\n" + "#" * 70)
print("TAHAP 2: CLEANING, DEDUPLIKASI, DAN GABUNG ATRIBUT OSM")
print("#" * 70)

print("\n-- Self-test fungsi normalisasi --")
self_test_normalisasi()

try:
    df_osm = pd.read_csv(OSM_FILE)
    print(f"\nOSM (mentah): {len(df_osm)} baris")
except FileNotFoundError:
    df_osm = pd.DataFrame(columns=["nama", "wifi", "outdoor_seating", "smoking"])
    print("\nOSM tidak ditemukan, dilewati.")

print("\n-- Ekstraksi koordinat --")
coords = df_cafe["link"].apply(extract_lat_lng)
df_cafe["latitude"] = coords.apply(lambda x: x[0])
df_cafe["longitude"] = coords.apply(lambda x: x[1])
print(f"Berhasil ekstrak koordinat: {df_cafe['latitude'].notna().sum()}/{len(df_cafe)}")

df_cafe["nama_normal"] = df_cafe["nama_cafe"].apply(normalize_name)
if "nama" in df_osm.columns:
    df_osm["nama_normal"] = df_osm["nama"].apply(normalize_name)

print("\n-- Deduplikasi cafe --")
before = len(df_cafe)
df_cafe = df_cafe.drop_duplicates(subset="nama_normal", keep="first").reset_index(drop=True)
print(f"Setelah dedup nama identik: {len(df_cafe)} (dari {before})")

to_drop = set()
records = df_cafe[["nama_normal", "latitude", "longitude"]].to_dict("records")
for i in range(len(records)):
    if i in to_drop:
        continue
    for j in range(i + 1, len(records)):
        if j in to_drop:
            continue
        a, b = records[i], records[j]
        if pd.isna(a["latitude"]) or pd.isna(b["latitude"]):
            continue
        jarak = haversine_km(a["latitude"], a["longitude"], b["latitude"], b["longitude"])
        kemiripan = fuzz.ratio(a["nama_normal"], b["nama_normal"])
        if jarak < 0.05 and kemiripan >= DEDUP_NAME_THRESHOLD:
            to_drop.add(j)

if to_drop:
    df_cafe = df_cafe.drop(df_cafe.index[list(to_drop)]).reset_index(drop=True)
print(f"Setelah dedup nama mirip + lokasi berdekatan: {len(df_cafe)} (menghapus {len(to_drop)} tambahan)")

print("\n-- Mencocokkan atribut dari OSM --")
df_cafe["wifi"] = None
df_cafe["outdoor_seating"] = None
df_cafe["smoking"] = None
jumlah_cocok = 0
if "nama_normal" in df_osm.columns and len(df_osm) > 0:
    for idx, row in df_cafe.iterrows():
        best_score, best_match = 0, None
        for _, osm_row in df_osm.iterrows():
            score = fuzz.ratio(row["nama_normal"], osm_row["nama_normal"])
            if score > best_score:
                best_score, best_match = score, osm_row
        if best_score >= OSM_MATCH_THRESHOLD:
            df_cafe.at[idx, "wifi"] = best_match.get("wifi")
            df_cafe.at[idx, "outdoor_seating"] = best_match.get("outdoor_seating")
            df_cafe.at[idx, "smoking"] = best_match.get("smoking")
            jumlah_cocok += 1
print(f"Cafe yang dilengkapi atribut dari OSM: {jumlah_cocok}/{len(df_cafe)}")

print("\n-- Menangani data kosong --")
df_cafe["harga_indikasi"] = df_cafe["harga_indikasi"].fillna("Tidak diketahui")
if "jam_operasional" in df_cafe.columns:
    df_cafe["jam_operasional"] = df_cafe["jam_operasional"].fillna("Tidak diketahui")
if "alamat" in df_cafe.columns:
    df_cafe["alamat"] = df_cafe["alamat"].fillna("Tidak diketahui")
df_cafe["rating_rata2"] = pd.to_numeric(df_cafe["rating_rata2"].astype(str).str.replace(",", "."), errors="coerce")
df_cafe["jumlah_ulasan_total"] = pd.to_numeric(df_cafe["jumlah_ulasan_total"], errors="coerce")

sebelum_buang = len(df_cafe)
df_cafe = df_cafe.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
print(f"Cafe tanpa koordinat valid dibuang: {sebelum_buang - len(df_cafe)}")

print("\n-- Cleaning ulasan --")
before = len(df_reviews)
df_reviews["ulasan"] = df_reviews["ulasan"].astype(str).str.strip()

# Urutan penting: emoji -> spasi, lalu huruf memanjang, lalu normalisasi singkatan
df_reviews["ulasan"] = df_reviews["ulasan"].apply(bersihkan_emoji)
setelah_emoji_kosong = (df_reviews["ulasan"].str.len() == 0).sum()
print(f"Ulasan yang jadi kosong setelah emoji dihapus (cuma berisi emoji): {setelah_emoji_kosong}")

# Huruf memanjang ("enakkk" -> "enak"), memakai kosakata korpus sendiri sebagai pembanding
vocab_korpus = bangun_vocab(df_reviews["ulasan"])
df_reviews["ulasan"] = df_reviews["ulasan"].apply(lambda t: perbaiki_pemanjangan(t, vocab_korpus))
df_reviews["ulasan"] = df_reviews["ulasan"].apply(normalisasi_singkatan)
df_reviews["ulasan"] = df_reviews["ulasan"].str.strip()

df_reviews = df_reviews[df_reviews["ulasan"].str.len() >= 3]
df_reviews = df_reviews.drop_duplicates(subset=["nama_cafe", "ulasan"], keep="first")
df_reviews["rating_bintang"] = pd.to_numeric(df_reviews["rating_bintang"].astype(str).str.replace(",", "."), errors="coerce")
cafe_valid = set(df_cafe["nama_cafe"])
df_reviews = df_reviews[df_reviews["nama_cafe"].isin(cafe_valid)].reset_index(drop=True)
print(f"Ulasan setelah cleaning: {len(df_reviews)} (dari {before})")

# ---- Diagnostik normalisasi ----
sisa = Counter()
token_pendek = Counter()
for u in df_reviews["ulasan"]:
    for t in re.findall(r"[A-Za-z]+", u.lower()):
        if t in KAMUS_NORMALISASI and KAMUS_NORMALISASI[t] != t:
            sisa[t] += 1
        if len(t) <= 4:
            token_pendek[t] += 1

print("\n-- Singkatan dari kamus yang masih tersisa (harusnya kosong) --")
print(sisa.most_common(20) if sisa else "Kosong (bagus)")
print("\n-- 80 token pendek paling sering (cek manual, cari singkatan yang belum ada di kamus) --")
print(token_pendek.most_common(80))

# Diagnostik tambahan: token 5-7 huruf (singkatan yang lebih panjang, mis. "sndiri", "bbrp")
token_sedang = Counter()
for u in df_reviews["ulasan"]:
    for t in re.findall(r"[A-Za-z]+", u.lower()):
        if 5 <= len(t) <= 7:
            token_sedang[t] += 1
print("\n-- 150 token 5-7 huruf paling sering (cek manual) --")
print(token_sedang.most_common(150))

# Diagnostik tambahan: pola aneh (huruf menempel angka, huruf berulang 3x atau lebih)
aneh = Counter()
for u in df_reviews["ulasan"]:
    for m in re.finditer(r"\b[A-Za-z]+\d\b|\b\w*(\w)\1{2,}\w*\b", u.lower()):
        aneh[m.group(0)] += 1
print("\n-- 30 pola aneh paling sering (angka menempel / huruf berulang) --")
print(aneh.most_common(30))

# Diagnostik tambahan: "nya" yang masih berdiri sendiri (lihat konteksnya)
nya_sisa = sum(1 for u in df_reviews["ulasan"] for _ in re.finditer(r"\bnya\b", u))
print(f"\n-- Kata 'nya' yang masih berdiri sendiri: {nya_sisa} kali. Contoh konteks: --")
contoh_nya = []
for u in df_reviews["ulasan"]:
    m = re.search(r".{0,15}\bnya\b.{0,8}", u)
    if m:
        contoh_nya.append(m.group(0))
    if len(contoh_nya) >= 15:
        break
for c in contoh_nya:
    print("   ", repr(c))

# Diagnostik tambahan: kandidat typo (hanya untuk ditinjau manual, tidak diubah otomatis)
print("\n-- Kandidat typo: kata jarang yang mirip kata yang jauh lebih sering (cek manual) --")
kata_sah = muat_kamus_kata_dasar()
if kata_sah:
    print(f"Kamus kata dasar dimuat ({len(kata_sah)} kata): kata sah tidak akan diusulkan sebagai typo.")
else:
    print("Kamus kata dasar tidak tersedia (pip install PySastrawi untuk menyaring kata sah dari daftar ini).")
kandidat_typo = cari_kandidat_typo(bangun_vocab(df_reviews["ulasan"]), kata_sah=kata_sah)
if kandidat_typo:
    pd.DataFrame(
        kandidat_typo,
        columns=["kata_diduga_typo", "freq_typo", "saran_perbaikan", "freq_saran"],
    ).to_csv(KANDIDAT_TYPO_OUT, index=False, encoding="utf-8-sig")
    for w, c, b, cb in kandidat_typo[:60]:
        print(f"   {w} ({c}x)  ->  {b} ({cb}x)")
    print(f"[i] {len(kandidat_typo)} kandidat disimpan ke {KANDIDAT_TYPO_OUT}.")
    print("    Salin ke KAMUS_TYPO HANYA kata yang memang bukan kata dan perbaikannya tidak ambigu.")
    print("    Jangan salin kata yang punya arti sendiri (mis. walau, tepat, taman), walau mirip kata lain.")
    print("    Cek juga kolom saran: bisa salah walau tokennya memang typo (mis. mayan bukan makan, tapi lumayan).")
else:
    print("Tidak ada kandidat.")

kolom_final_cafe = [c for c in [
    "nama_cafe", "link", "latitude", "longitude", "alamat", "rating_rata2",
    "jumlah_ulasan_total", "harga_indikasi", "jam_operasional",
    "wifi", "outdoor_seating", "smoking",
] if c in df_cafe.columns]

df_cafe[kolom_final_cafe].to_csv(CAFE_CLEAN_OUT, index=False, encoding="utf-8-sig")
df_reviews.to_csv(REVIEWS_CLEAN_OUT, index=False, encoding="utf-8-sig")
print(f"\nCafe bersih   : {len(df_cafe)} -> {CAFE_CLEAN_OUT}")
print(f"Ulasan bersih : {len(df_reviews)} -> {REVIEWS_CLEAN_OUT}")


# ==================== TAHAP 3: VALIDASI ====================

print("\n" + "#" * 70)
print("TAHAP 3: VALIDASI KUALITAS DATA")
print("#" * 70)

df_cafe_v = pd.read_csv(CAFE_CLEAN_OUT)
df_reviews_v = pd.read_csv(REVIEWS_CLEAN_OUT)

print("\n=== 1. JUMLAH DATA ===")
print(f"Total cafe   : {len(df_cafe_v)}")
print(f"Total ulasan : {len(df_reviews_v)}")
print(f"Rata-rata ulasan per cafe : {len(df_reviews_v) / len(df_cafe_v):.1f}")

print("\n=== 2. DATA KOSONG PER KOLOM ===")
print("-- cafe_clean.csv --")
print(df_cafe_v.isna().sum())

print("\n=== 3. DUPLIKAT TERSISA ===")
print(f"Nama cafe masih duplikat: {df_cafe_v['nama_cafe'].duplicated().sum()} (harusnya 0)")

print("\n=== 4. RENTANG NILAI ===")
print(f"Rating: {df_cafe_v['rating_rata2'].min()} - {df_cafe_v['rating_rata2'].max()}")
print(f"Latitude: {df_cafe_v['latitude'].min()} - {df_cafe_v['latitude'].max()}")
print(f"Longitude: {df_cafe_v['longitude'].min()} - {df_cafe_v['longitude'].max()}")

print("\n=== 5. CAKUPAN ULASAN ===")
jumlah_per_cafe = df_reviews_v.groupby("nama_cafe").size().reset_index(name="ulasan_terambil")
cek = df_cafe_v.merge(jumlah_per_cafe, on="nama_cafe", how="left")
cek["ulasan_terambil"] = cek["ulasan_terambil"].fillna(0).astype(int)
cek["persentase_cakupan"] = (cek["ulasan_terambil"] / cek["jumlah_ulasan_total"] * 100).round(1)
print(f"Cafe tanpa ulasan sama sekali: {(cek['ulasan_terambil'] == 0).sum()}")
print(f"Rata-rata cakupan: {cek['persentase_cakupan'].mean():.1f}%")
print("\n5 cafe cakupan terendah:")
print(cek.nsmallest(5, "persentase_cakupan")[["nama_cafe", "jumlah_ulasan_total", "ulasan_terambil", "persentase_cakupan"]].to_string())

cek.to_csv(CEK_CAKUPAN_OUT, index=False, encoding="utf-8-sig")
print(f"\n[i] Detail cakupan disimpan ke {CEK_CAKUPAN_OUT}")

print("\n" + "#" * 70)
print("SEMUA TAHAP SELESAI")
print("#" * 70)

######################################################################
TAHAP 1: MERGE HASIL RETRY KE DATA MENTAH UTAMA
######################################################################
Data cafe mentah (sebelum merge retry)   : 446
Data ulasan mentah (sebelum merge retry) : 72915
Ditemukan hasil retry: 19 cafe, 1539 ulasan
Setelah merge retry -> cafe: 446, ulasan: 72915
(file_cafe_lengkap.csv dan file_google_maps_reviews.csv sudah diperbarui)

######################################################################
TAHAP 2: CLEANING, DEDUPLIKASI, DAN GABUNG ATRIBUT OSM
######################################################################

-- Self-test fungsi normalisasi --
Self-test normalisasi: 22/22 lulus
Self-test emoji: 'enak😍yg bagus' -> 'enak yang bagus' (harusnya 'enak yang bagus')

OSM (mentah): 74 baris

-- Ekstraksi koordinat --
Berhasil ekstrak koordinat: 446/446

-- Deduplikasi cafe --
Setelah dedup nama identik: 440 (dari 446)
Setelah dedup nama mirip + lokasi berdekat